# 02 — VoC Theme Analysis

Explore theme frequency, sentiment (negative share), satisfaction impact, and detractor over-index.

**Core logic:** `src/classify_voc_themes.py`, `src/analyze_themes.py` → outputs in `outputs/tables/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from config import VOC_THEMES

TABLES = ROOT / "outputs" / "tables"
BRAND = "#6F4E37"

In [ ]:
theme_summary = pd.read_csv(TABLES / "theme_summary.csv")
theme_impact = pd.read_csv(TABLES / "theme_impact.csv")
detractor = pd.read_csv(TABLES / "detractor_theme_analysis.csv")

print(f"Themes tracked: {len(VOC_THEMES)}")
theme_summary.head()

## Theme frequency

Comment volume and share of classified feedback by primary theme.

In [ ]:
freq = theme_summary.sort_values("comment_count", ascending=True).copy()
freq["theme_label"] = freq["primary_theme"].str.replace("_", " ").str.title()

fig = px.bar(
    freq,
    x="comment_count",
    y="theme_label",
    orientation="h",
    title="Theme Frequency (Comment Volume)",
    color_discrete_sequence=[BRAND],
)
fig.show()

In [ ]:
theme_summary[["primary_theme", "comment_count", "share_of_comments"]].head(5)

## Theme sentiment

Negative comment share and detractor rate by theme (proxy for sentiment mix).

In [ ]:
sent = theme_summary.sort_values("negative_share", ascending=True).copy()
sent["theme_label"] = sent["primary_theme"].str.replace("_", " ").str.title()

fig = px.bar(
    sent,
    x="negative_share",
    y="theme_label",
    orientation="h",
    title="Negative Comment Share by Theme",
    color="negative_share",
    color_continuous_scale=["#F5F5F5", "#C0392B"],
)
fig.update_layout(showlegend=False)
fig.show()

## NPS / CSAT / revisit impact by theme

Theme-level averages and gaps vs brand baseline (`theme_impact.csv`).

In [ ]:
impact_cols = [
    "primary_theme", "theme_avg_nps", "theme_nps_gap",
    "theme_avg_csat", "theme_csat_gap",
    "theme_avg_revisit_intent", "theme_revisit_gap", "impact_rank",
]
theme_impact.sort_values("impact_rank")[impact_cols]

In [ ]:
scatter = theme_summary.merge(
    theme_impact[["primary_theme", "theme_nps_gap", "theme_revisit_gap"]],
    on="primary_theme",
)
scatter["theme_label"] = scatter["primary_theme"].str.replace("_", " ").str.title()

fig = px.scatter(
    scatter,
    x="share_of_comments",
    y="theme_nps_gap",
    size="negative_share",
    color="theme_revisit_gap",
    hover_name="theme_label",
    title="Theme Frequency vs NPS Gap (size = negative share)",
    labels={"share_of_comments": "Comment share", "theme_nps_gap": "NPS gap vs brand"},
)
fig.show()

## Detractor over-index

Themes that appear disproportionately among detractor comments (`detractor_over_index` > 1).

In [ ]:
det = detractor.sort_values("detractor_over_index", ascending=True).copy()
det["theme_label"] = det["primary_theme"].str.replace("_", " ").str.title()

fig = px.bar(
    det,
    x="detractor_over_index",
    y="theme_label",
    orientation="h",
    title="Detractor Over-Index by Theme (1.0 = proportional)",
    color_discrete_sequence=[BRAND],
)
fig.add_vline(x=1.0, line_dash="dash", annotation_text="baseline")
fig.show()

In [ ]:
detractor.sort_values("detractor_over_index", ascending=False)[
    ["primary_theme", "share_of_all_comments", "share_of_detractor_comments", "detractor_over_index", "negative_share"]
].head(5)

## Insight

**Speed of service** leads comment volume, but **price/value** shows the deepest NPS gap. High frequency does not always equal highest loyalty risk — compare volume charts with gap and detractor over-index tables before prioritizing.